In [1]:
from transformers import AutoTokenizer, AutoModel
from bertviz import head_view

/home/russele7/practicum/dle/sprint_5/venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
model_name = "bert-base-multilingual-cased"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name, output_attentions=True)

In [3]:
sentence = "Иван Иванович Иванов работает в Газпроме."
inputs = tokenizer.encode(sentence, return_tensors='pt')
outputs = model(inputs)
attention = outputs.attentions  # список матриц внимания по слоям

In [4]:
# Покажем head-view для слоя 0, голова 0 (пример)
head_view(attention, tokenizer.convert_ids_to_tokens(inputs[0]), layer=0, heads=0) 

<IPython.core.display.Javascript object>

# Task 1

In [5]:
from transformers import AutoModel, AutoTokenizer
from bertviz import head_view
import torch

In [6]:
# Модель для анализа attention (без головы классификации)
model_name = "cointegrated/rubert-tiny2"
attention_model = AutoModel.from_pretrained(model_name, output_attentions=True) # Ваш код здесь
tokenizer = AutoTokenizer.from_pretrained(model_name) # Ваш код здесь

In [7]:
def analyze_attention(sentence, layer=0, heads=0):
    """Анализирует attention для заданного предложения"""
    inputs = tokenizer(sentence, return_tensors='pt')
    tokens = tokenizer.convert_ids_to_tokens(inputs['input_ids'][0])

    print(f"Анализируем предложение: {sentence}")
    print(f"Токены: {tokens}")
    
    with torch.no_grad():
        # Получите outputs
        outputs = attention_model(**inputs) # Ваш код здесь
        # Получите attentions из outputs
        attentions = outputs.attentions # Ваш код здесь
    
    print(f"Количество слоёв: {len(attentions)}")    
    # Визуализация конкретной головы
    # Ваш код здесь
    head_view(attentions, tokens, layer=layer, heads=[heads]) 

In [8]:
example_sentence = "Иван Петров работает в Google."
# Ваш код здесь
analyze_attention(sentence, layer=0, heads=0)

Анализируем предложение: Иван Иванович Иванов работает в Газпроме.
Токены: ['[CLS]', 'Иван', 'Иванович', 'Иванов', 'работает', 'в', 'Газпром', '##е', '.', '[SEP]']
Количество слоёв: 3


<IPython.core.display.Javascript object>

In [9]:
example_sentence = "Иван Иванович Иванов работает в Газпроме."
analyze_attention(example_sentence, layer=2, heads=3) 

Анализируем предложение: Иван Иванович Иванов работает в Газпроме.
Токены: ['[CLS]', 'Иван', 'Иванович', 'Иванов', 'работает', 'в', 'Газпром', '##е', '.', '[SEP]']
Количество слоёв: 3


<IPython.core.display.Javascript object>

# Task 2

In [10]:
from corus import load_factru
import re

In [11]:
dir_path = "factRuEval-2016/" 

In [12]:
# Загрузите записи
records = list(load_factru(dir_path))
print("Загружено записей:", len(records))

Загружено записей: 254


In [13]:
# Реализуйте whitespace-tokenizer
def whitespace_tokenize_with_offsets(text: str):
    # Ваш код здесь
    # должен возвращать: tokens (list[str]), spans (list[(start,end)])
    tokens = []
    spans = []
    for m in re.finditer(r'\S+', text):
        tokens.append(m.group())
        spans.append((m.start(), m.end()))

    return tokens, spans

# Task 3

In [14]:
records[-1]

FactruMarkup(
    id='3878',
    text='Скотланд-Ярд вызвал на допрос Руперта Мердока\n\n83-летний медиамагнат является подозреваемым по делу о нарушении закона принадлежащими ему британскими газетами.\n\nЛондонская полиция — Скотланд-Ярд — официально вызвала 83-летнего медиамагната Руперта Мердока (№78 в глобальном рейтинге миллиардеров, доход семьи — 14,2 млрд долларов) на допрос в качестве подозреваемого по делу о нарушении закона принадлежащими ему британскими газетами.\n\nВ первый раз полиция уведомила об этом Мердока в 2013 году. Однако тогда с адвокатами медиамагната было достигнуто соглашение, что дальнейшие действия последуют после завершения процесса по делу о прослушке телефонных линий знаменитостей, политиков и членов королевских семей журналистами еженедельника News of the World. Во вторник, 24 июня, суд признал признал виновным по делу о прослушивании редактора таблоида Энди Коулсон. В то же время, все обвинения с помощницы Мердока и главы издательства News International б

In [15]:
unique_labels = set()
for record in records:
    for obj in record.objects:
        unique_labels.add(obj.type)
print("Уникальные метки в исходном датасете:", unique_labels) 

Уникальные метки в исходном датасете: {'Facility', 'Org', 'Location', 'Project', 'Person', 'LocOrg'}


In [16]:
def map_object_type(obj_type: str) -> str:
    """
    Сопоставляет строковое обозначение типа объекта (obj_type) к одной из базовых меток:
    {'PER', 'ORG', 'LOC', 'MISC'}.

    Вход:
        obj_type (str): строка с исходным типом/подтипом сущности из разметки FactRu.
                        Может быть в разном регистре и содержать подтипы, например:
                        "person", "PER.NAME", "Organization", "GPE", "LOC_city", "DATE", "product" и т.д.

    Возвращает:
        str: одна из {'PER','ORG','LOC','MISC'}.

    Примеры:
        map_object_type("PER") -> "PER"
        map_object_type("person_name") -> "PER"
        map_object_type("organization") -> "ORG"
        map_object_type("GPE") -> "LOC"
        map_object_type("event") -> "MISC"
    """ 
    t = (obj_type or "").lower()
    if "person" in t or t in {"person", "name", "surname", "firstname", "patronymic"}:
        return "PER"
    if "org" in t or "organization" in t or "company" in t or "org_name" in t or "org_descr" in t:
        return "ORG"
    if "loc" in t or "location" in t or "geo" in t or "place" in t or "loc_name" in t:
        return "LOC"
    return "MISC" 
    

# Task 4

In [17]:
records[-1].objects[1]

FactruObject(
    id='50053',
    type='Person',
    spans=[FactruSpan(
         id='83697',
         type='name',
         start=30,
         stop=37
     ),
     FactruSpan(
         id='83698',
         type='surname',
         start=38,
         stop=45
     )]
)

In [18]:
text = records[-1].text
tokens, token_spans = whitespace_tokenize_with_offsets(text)
token_labels = ["O"] * len(tokens)

In [19]:
examples = []
for rec in records:
    text = rec.text
    tokens, token_spans = whitespace_tokenize_with_offsets(text)
    token_labels = ["O"] * len(tokens)

    # Пройдите по rec.objects и заполните token_labels
    # Ваш код здесь
    for obj in rec.objects:
        base_type = map_object_type(obj.type)
        for span in obj.spans:
            span_start = span.start
            span_end = span.stop
            overlapping_idxs = []
            for i, (t_start, t_end) in enumerate(token_spans):
                if not (t_end <= span_start or t_start >= span_end):
                    overlapping_idxs.append(i)
            if not overlapping_idxs:
                # можно логировать: print(f"No overlap for span {span_start}-{span_end} in doc {rec.id}")
                continue
            for j, tok_idx in enumerate(overlapping_idxs):
                if token_labels[tok_idx] != "O":
                    continue
                prefix = "B" if j == 0 else "I"
                token_labels[tok_idx] = f"{prefix}-{base_type}"

    examples.append({
        "id": rec.id,
        "text": rec.text,
        "tokens": tokens,
        "tags": token_labels
    })

print(f"Примеры собраны: {len(examples)}")
# Посмотрим пример
print("Пример tokens/tags:", examples[2]["tokens"][:20], examples[2]["tags"][:20]) 

Примеры собраны: 254
Пример tokens/tags: ['Юрий', 'Маленченко', 'и', 'Пегги', 'Уитсон,', 'основной', 'состав', '16-й', 'долговременной', 'экспедиции', 'МКС,', 'и', 'первая', 'корейская', 'женщина-космонавт', 'Ли', 'Со', 'Ён', 'вернулись', 'на'] ['B-PER', 'B-PER', 'O', 'B-PER', 'B-PER', 'O', 'O', 'B-ORG', 'I-ORG', 'I-ORG', 'B-ORG', 'O', 'O', 'O', 'O', 'B-PER', 'B-PER', 'I-PER', 'O', 'O']


In [20]:
print(examples[-1])

{'id': '3878', 'text': 'Скотланд-Ярд вызвал на допрос Руперта Мердока\n\n83-летний медиамагнат является подозреваемым по делу о нарушении закона принадлежащими ему британскими газетами.\n\nЛондонская полиция — Скотланд-Ярд — официально вызвала 83-летнего медиамагната Руперта Мердока (№78 в глобальном рейтинге миллиардеров, доход семьи — 14,2 млрд долларов) на допрос в качестве подозреваемого по делу о нарушении закона принадлежащими ему британскими газетами.\n\nВ первый раз полиция уведомила об этом Мердока в 2013 году. Однако тогда с адвокатами медиамагната было достигнуто соглашение, что дальнейшие действия последуют после завершения процесса по делу о прослушке телефонных линий знаменитостей, политиков и членов королевских семей журналистами еженедельника News of the World. Во вторник, 24 июня, суд признал признал виновным по делу о прослушивании редактора таблоида Энди Коулсон. В то же время, все обвинения с помощницы Мердока и главы издательства News International были сняты.\n\nР

In [21]:
from datasets import Dataset, DatasetDict

In [22]:
unique_labels = set()
for ex in examples:
    unique_labels.update(ex["tags"])
unique_labels.add("O")
label_list = sorted(unique_labels)
label2id = {lab: i for i, lab in enumerate(label_list)}
id2label = {i: lab for lab, i in label2id.items()}

for ex in examples:
    ex["tags"] = [label2id[t] for t in ex["tags"]]

full_ds = Dataset.from_list(examples)
split = full_ds.train_test_split(test_size=0.1, seed=42)
dataset = DatasetDict({"train": split["train"], "test": split["test"]})
print(dataset) 

DatasetDict({
    train: Dataset({
        features: ['id', 'text', 'tokens', 'tags'],
        num_rows: 228
    })
    test: Dataset({
        features: ['id', 'text', 'tokens', 'tags'],
        num_rows: 26
    })
})


# Task 5

In [23]:
def tokenize_and_align_labels(examples_batch):
    tokenized = tokenizer(
        examples_batch["tokens"],
        is_split_into_words=True,
        truncation=True,
        padding="max_length",
        max_length=128
    )
    labels = []
    for i, word_labels in enumerate(examples_batch["tags"]):
        word_ids = tokenized.word_ids(batch_index=i)
        label_ids = []
        prev_word_idx = None
        for word_idx in word_ids:
            # Условие: если word_idx == None, то это padding/special token
            # Иначе, если word_idx != prev_word_idx, то это начало нового слова
            # Ваш код здесь
            if word_idx is None:
                label_ids.append(-100)
            elif word_idx != prev_word_idx:
                label_ids.append(word_labels[word_idx])
            else:
                label_ids.append(-100)
            prev_word_idx = word_idx
        labels.append(label_ids)
    tokenized["labels"] = labels
    return tokenized

In [24]:
# 8. Применяем токенизацию к датасету
tokenized_dataset = dataset.map(
    tokenize_and_align_labels,
    batched=True,
    remove_columns=["text", "tokens", "tags", "id"]
)

Map: 100%|██████████| 26/26 [00:00<00:00, 1036.03 examples/s]


In [25]:
print(tokenized_dataset)

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 228
    })
    test: Dataset({
        features: ['input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 26
    })
})


In [26]:
# Проверка: показываем один пример из train
print(tokenized_dataset["train"][0])

{'input_ids': [2, 57429, 1044, 3058, 2356, 40486, 22789, 33500, 105, 45543, 72562, 18125, 117, 733, 51298, 44810, 36097, 41272, 314, 37860, 35778, 2445, 833, 2480, 18, 20260, 2215, 49091, 5731, 36207, 329, 35457, 57306, 314, 32381, 23064, 105, 45543, 72562, 18125, 117, 320, 1142, 46743, 322, 55624, 15107, 44050, 15676, 28625, 5064, 16, 31102, 39344, 19195, 17, 33935, 18, 282, 4469, 10173, 2520, 16, 56311, 16, 48907, 49581, 9780, 320, 36897, 45299, 43977, 2647, 48250, 71801, 16794, 9970, 18, 36097, 41272, 314, 37860, 35778, 2445, 833, 2480, 18, 105, 38104, 6464, 48029, 2356, 40855, 34911, 40052, 794, 35671, 15213, 40654, 32843, 3092, 78172, 1854, 16794, 9970, 320, 1870, 46743, 18, 282, 4859, 32973, 16, 20353, 16, 41402, 4544, 21651, 47822, 17117, 329, 769, 39110, 11420, 4337, 35164, 16, 3], 'token_type_ids': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 

# Task 6

In [27]:
from transformers import DataCollatorForTokenClassification
from torch.utils.data import DataLoader

In [28]:
# Предполагается, что tokenized_dataset уже создан (после dataset.map(tokenize_and_align_labels))
# Ваш код здесь: создайте data_collator, train_dataloader
data_collator = DataCollatorForTokenClassification(tokenizer)

train_dataloader = DataLoader(
    tokenized_dataset["train"],
    batch_size=16,
    shuffle=True,
    collate_fn=data_collator
)

In [29]:
print("Готово. Примеры для обучения:", len(tokenized_dataset["train"])) 

Готово. Примеры для обучения: 228


# Task 7

In [30]:
# Функция, выравнивающая предсказания модели и реальные метки (на уровне tokenized_dataset)
def get_flat_labels_and_preds_from_model(tokenized_split, model, device, max_samples=None):
    """
    tokenized_split: dataset split (list-like of examples with keys 'input_ids','attention_mask','labels')
    Возвращает flat lists: y_true (ints), y_pred (ints)
    """
    y_true = []
    y_pred = []
    for i, ex in enumerate(tokenized_split):
        if max_samples is not None and i >= max_samples:
            break

        # Превращаем в тензоры (batch size = 1)
        input_ids = torch.tensor([ex["input_ids"]], dtype=torch.long).to(device)
        attention_mask = torch.tensor([ex["attention_mask"]], dtype=torch.long).to(device)

        with torch.no_grad():
            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            logits = outputs.logits  # shape (1, seq_len, num_labels)
            preds = torch.argmax(logits, dim=-1).squeeze(0).cpu().tolist()  # list длины seq_len

        # Истинные метки (включая -100 для пэддинга/ignored)
        true_labels = ex["labels"]  # список длиной seq_len; элементы -100 или id

        # Фильтруем позиции, где true != -100
        filtered_true = []
        filtered_pred = []
        for p, t in zip(preds, true_labels):
            if t == -100:
                continue
            filtered_true.append(int(t))
            filtered_pred.append(int(p))

        # Обрежем на минимальную длину (на случай рассинхронизации)
        minlen = min(len(filtered_true), len(filtered_pred))
        if minlen == 0:
            continue
        y_true.extend(filtered_true[:minlen])
        y_pred.extend(filtered_pred[:minlen])

    return y_true, y_pred

In [31]:
from transformers import AutoModelForTokenClassification
from sklearn.metrics import precision_score, recall_score, f1_score

In [32]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Загружаем модель 
model = AutoModelForTokenClassification.from_pretrained(
    "cointegrated/rubert-tiny2",
    num_labels=len(label2id),
    id2label=id2label,
    label2id=label2id
)

Some weights of BertForTokenClassification were not initialized from the model checkpoint at cointegrated/rubert-tiny2 and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [33]:
model.to(device)
model.eval()

BertForTokenClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(83828, 312, padding_idx=0)
      (position_embeddings): Embedding(2048, 312)
      (token_type_embeddings): Embedding(2, 312)
      (LayerNorm): LayerNorm((312,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-2): 3 x BertLayer(
          (attention): BertAttention(
            (self): BertSdpaSelfAttention(
              (query): Linear(in_features=312, out_features=312, bias=True)
              (key): Linear(in_features=312, out_features=312, bias=True)
              (value): Linear(in_features=312, out_features=312, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=312, out_features=312, bias=True)
              (LayerNorm): LayerNorm((312,), eps=1e-12,

In [34]:
# Получите token-level предсказания и истинные метки с помощью готовой функции
# (max_samples можно уменьшить/увеличить по ресурсам)
y_true, y_pred = get_flat_labels_and_preds_from_model(tokenized_dataset["test"], model, device, max_samples=200) # Ваш код здесь

In [35]:
print("Samples used (token-level):", len(y_true))
print("Precision:", precision_score(y_true, y_pred, average="macro", zero_division=0))
print("Recall:   ", recall_score(y_true, y_pred, average="macro", zero_division=0))
print("F1:       ", f1_score(y_true, y_pred, average="macro", zero_division=0)) 

Samples used (token-level): 2295
Precision: 0.11689228881628098
Recall:    0.0913138401521253
F1:        0.05801999592597462


# Task 8

In [36]:
from transformers import DataCollatorForTokenClassification
from torch.utils.data import DataLoader
import torch
from tqdm import tqdm

In [41]:
# Параметры
num_epochs = 50       # уменьшите при необходимости
learning_rate = 5e-5


# 2) Оптимизатор
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

In [42]:

# 3) Loop обучения
model.train()
for epoch in range(num_epochs):
    total_loss = 0.0
    n_batches = 0
    for batch in tqdm(train_dataloader, desc=f"Epoch {epoch+1}"):
        # Перенести batch на device, вычислить loss, backward, step, zero_grad
        # Ваш код здесь
        # batch = batch.to(device)
        batch = {k: v.to(device) for k, v in batch.items()}
        outputs = model(**batch)
        loss = outputs.loss
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()
        total_loss += loss.item()
        n_batches += 1

    avg_loss = total_loss / n_batches if n_batches > 0 else 0.0
    print(f"Epoch {epoch+1} avg loss: {avg_loss:.4f}")
    print(f"Epoch {epoch+1} avg loss: {total_loss/len(train_dataloader):.4f}")

Epoch 1: 100%|██████████| 15/15 [00:00<00:00, 22.16it/s]


Epoch 1 avg loss: 0.3425
Epoch 1 avg loss: 0.3425


Epoch 2: 100%|██████████| 15/15 [00:00<00:00, 26.62it/s]


Epoch 2 avg loss: 0.2790
Epoch 2 avg loss: 0.2790


Epoch 3: 100%|██████████| 15/15 [00:00<00:00, 26.77it/s]


...

Epoch 48 avg loss: 0.0074
Epoch 48 avg loss: 0.0074


Epoch 49: 100%|██████████| 15/15 [00:00<00:00, 30.08it/s]


Epoch 49 avg loss: 0.0063
Epoch 49 avg loss: 0.0063


Epoch 50: 100%|██████████| 15/15 [00:00<00:00, 29.88it/s]

Epoch 50 avg loss: 0.0062
Epoch 50 avg loss: 0.0062


In [43]:
# 4) Оценка после fine-tune
# Переводим модель в eval перед оценкой
model.eval()
y_true_ft, y_pred_ft = get_flat_labels_and_preds_from_model(
    tokenized_dataset["test"], model, device, max_samples=200
)

In [44]:
print("After fine-tuning:")
print("Precision:", precision_score(y_true_ft, y_pred_ft, average="macro", zero_division=0))
print("Recall:   ", recall_score (y_true_ft, y_pred_ft, average="macro", zero_division=0))
print("F1:       ", f1_score   (y_true_ft, y_pred_ft, average="macro", zero_division=0))

# 5) Сравнение с baseline (предполагается, что baseline метрики сохранены)
# Ваш код: загрузите baseline F1 и выведите delta
# Ваш код здесь

After fine-tuning:
Precision: 0.625739071595609
Recall:    0.5729845054217081
F1:        0.5910255629167503
